# AI Code Documenter Solution designed with Gradio UI— Gradio

This solution adds **docstrings & inline comments** to Python code using any of your configured LLMs — OpenAI, Claude, Gemini, Grok, Groq, OpenRouter, or local Ollama. And integrated to beautiful Gradio interface. 
This solution gives you the flexibility of copying and pasting your code to the gradio UI or uploading a python file from your computer. 

## 1. Install Dependencies

In [ ]:
!pip install openai gradio requests python-dotenv --quiet
print("✅ All dependencies installed")

## 2. Load API Keys & Instantiate Clients

In [ ]:
import os
import re
import requests
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

openai_api_key     = os.getenv('OPENAI_API_KEY')
anthropic_api_key  = os.getenv('ANTHROPIC_API_KEY')
google_api_key     = os.getenv('GOOGLE_API_KEY')
grok_api_key       = os.getenv('GROK_API_KEY')
groq_api_key       = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


#  Set you Base URLs and models to use
grok_url       = "https://api.x.ai/v1"
groq_url       = "https://api.groq.com/openai/v1"
ollama_url     = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

openai     = OpenAI(api_key=openai_api_key)
grok       = OpenAI(api_key=grok_api_key,       base_url=grok_url)
groq       = OpenAI(api_key=groq_api_key,       base_url=groq_url)
ollama     = OpenAI(api_key="ollama",            base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


models = [ "gpt-4o", "anthropic/claude-sonnet-4-5", "google/gemini-2.5-pro", "x-ai/grok-4", "openai/gpt-oss-120b", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b:instruct", ]


clients = {"gpt-4o": openai,
    "anthropic/claude-sonnet-4-5": openrouter,
    "google/gemini-2.5-pro": openrouter,
    "x-ai/grok-4": openrouter,
    "openai/gpt-oss-120b": groq,
    "qwen2.5-coder": ollama,
    "deepseek-coder-v2": ollama,
    "gpt-oss:20b": ollama,
    "qwen/qwen3-coder-30b-a3b:instruct": openrouter,
}

# Fetches Ollama models 
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODELS   = [m for m, c in clients.items() if c is ollama]

def get_ollama_models() -> list:
    """Fetch locally installed Ollama models; fall back to client map."""
    try:
        resp = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3)
        if resp.ok:
            names = [m["name"] for m in resp.json().get("models", [])]
            return names or OLLAMA_MODELS
    except Exception:
        pass
    return OLLAMA_MODELS

print("✅ Clients and models ready")

## 3. Documenter Core

In [4]:
STYLE_INSTRUCTIONS = {
    "Google-style": "Use Google-style docstrings with Args / Returns / Raises sections.",
}

BASE_SYSTEM_PROMPT = """You are an expert Python documentation assistant.

Your task:
1. Add docstrings to ALL functions and classes that are missing one.
2. Insert brief inline # comments on complex or non-obvious lines.

Strict rules:
- Preserve ALL original code exactly — logic, whitespace, naming, structure.
- Only INSERT docstrings/comments; never rewrite or reformat existing code.
- Add inline comments only where genuinely helpful; skip obvious lines.
- Return ONLY the documented Python code — no markdown fences, no explanation."""


def strip_fences(text: str) -> str:
    """Remove optional ```python ... ``` fences some models add."""
    text = re.sub(r"^```[\w]*\n?", "", text.strip())
    text = re.sub(r"\n?```$",      "", text.strip())
    return text.strip()


def document_code(code: str, model: str, doc_style: str = "Google-style") -> str:
    """
    Send code to the selected model via its mapped client and return documented version.

    Args:
        code:      Raw Python source.
        model:     Model name — must exist as a key in `clients`.
        doc_style: One of 'Google-style', 'NumPy-style', 'reStructuredText'.

    Returns:
        Documented Python source code as a string.

    Raises:
        ValueError: If model is not in the clients map or no code is provided.
    """
    if not code.strip():
        raise ValueError("No code provided.")
    if model not in clients:
        raise ValueError(f"Model '{model}' not found in clients map.")

    style_note = STYLE_INSTRUCTIONS.get(doc_style, STYLE_INSTRUCTIONS["Google-style"])
    prompt     = (
        f"{BASE_SYSTEM_PROMPT}\n{style_note}\n\n"
        f"Here is the Python code:\n\n{code}\n\n"
        "Return the documented code now:"
    )

    client = clients[model]
    resp   = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=4096,
    )
    return strip_fences(resp.choices[0].message.content)


print("✅ Documenter core ready")

✅ Documenter core ready


## 4. Gradio Interface

In [5]:
import gradio as gr

SAMPLE_CODE = '''\
def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    seq = [0, 1]
    while len(seq) < n:
        seq.append(seq[-1] + seq[-2])
    return seq


class Stack:
    def __init__(self):
        self._data = []

    def push(self, item):
        self._data.append(item)

    def pop(self):
        if not self._data:
            raise IndexError("pop from empty stack")
        return self._data.pop()

    def peek(self):
        return self._data[-1] if self._data else None
'''

CSS = """
.footer-note { font-size: 0.8rem; color: #888; margin-top: 1rem; }
"""

FOOTER_HTML = """
<p class="footer-note">
  Claude / Gemini / Grok / Qwen &rarr; OpenRouter &nbsp;&middot;&nbsp;
  gpt-oss-120b &rarr; Groq &nbsp;&middot;&nbsp;
  qwen2.5-coder / deepseek-coder-v2 / gpt-oss:20b &rarr; Ollama
</p>
"""

# Build routing reference table
CLIENT_LABELS = {
    id(openai):     "OpenAI",
    id(openrouter): "OpenRouter",
    id(groq):       "Groq",
    id(grok):       "Grok (xAI)",
    id(ollama):     "Ollama",
}

routing_rows = "\n".join(
    f"| `{m}` | {CLIENT_LABELS.get(id(clients[m]), '?')} |"
    for m in models
)
ROUTING_TABLE = f"| Model | Backend |\n|---|---|\n{routing_rows}"


# Gradio callbacks

def run_documenter(model, code_text, uploaded_file):
    """
    Gradio handler: document code and return result + status message.

    Args:
        model:         Selected model name from the dropdown.
        code_text:     Raw code pasted in the text area.
        uploaded_file: Gradio filepath string for optional .py upload.
        doc_style:     Docstring style selector (hardcoded to Google-style).

    Returns:
        Tuple of (documented_code: str, status_message: str).
    """
    # uploaded python file
    if uploaded_file is not None:
        code = Path(uploaded_file).read_text(encoding="utf-8")
    else:
        code = code_text

    if not code.strip():
        return "", "⚠️  Please paste code or upload a .py file."

    try:
        result    = document_code(code, model, "Google-style")
        lines_in  = len(code.splitlines())
        lines_out = len(result.splitlines())
        added     = lines_out - lines_in
        backend   = CLIENT_LABELS.get(id(clients[model]), "?")
        status    = (
            f"✅  Done — {model} via {backend}  "
            f"| {lines_in} → {lines_out} lines  "
            f"| +{added} lines added"
        )
        return result, status

    except Exception as e:
        return "", f"❌  Error: {e}"


def load_file_to_editor(file):
    """Read uploaded .py file into the code editor textarea."""
    if file is None:
        return ""
    return Path(file).read_text(encoding="utf-8")


# Gradio app

with gr.Blocks(css=CSS, title="AI Code Documenter") as demo:

    gr.Markdown("## AI Code Documenter")

    with gr.Row():
       
        with gr.Column(scale=1, min_width=280):
            model_dd = gr.Dropdown(
                choices=models,
                value=models[0],
                label="Model",
                interactive=True,
            )

            upload_file = gr.File(
                label="Upload .py file (optional)",
                file_types=[".py"],
                type="filepath",
            )

            run_btn = gr.Button(
                "Document Code",
                variant="primary",
            )

            status_out = gr.Textbox(
                label="Status",
                interactive=False,
                lines=2,
            )

            gr.Markdown(ROUTING_TABLE)

      
        with gr.Column(scale=3):
            with gr.Tabs():
                with gr.TabItem("📝  Input Code"):
                    code_in = gr.Code(
                        value=SAMPLE_CODE,
                        language="python",
                        label="Paste your Python code here",
                        lines=28,
                        interactive=True,
                    )
                with gr.TabItem("✅  Documented Output"):
                    code_out = gr.Code(
                        language="python",
                        label="Documented code",
                        lines=28,
                        interactive=False,
                    )

    gr.HTML(FOOTER_HTML)

 
    upload_file.change(
        fn=load_file_to_editor,
        inputs=upload_file,
        outputs=code_in,
    )

    run_btn.click(
        fn=run_documenter,
        inputs=[model_dd, code_in, upload_file],
        outputs=[code_out, status_out],
    )


print("✅ Gradio app sucessfully built")

✅ Gradio app sucessfully built


## 5. Launch

In [ ]:
demo.launch(
    inbrowser=True,
)

---
## Quick Reference

| Model | Routed via | Requires |
|---|---|---|
| `gpt-4o` | OpenAI direct | `OPENAI_API_KEY` |
| `anthropic/claude-sonnet-4-5` | OpenRouter | `OPENROUTER_API_KEY` |
| `google/gemini-2.5-pro` | OpenRouter | `OPENROUTER_API_KEY` |
| `x-ai/grok-4` | OpenRouter | `OPENROUTER_API_KEY` |
| `openai/gpt-oss-120b` | Groq | `GROQ_API_KEY` |
| `qwen2.5-coder`, `deepseek-coder-v2`, `gpt-oss:20b` | Ollama local | `ollama serve` on :11434 |
| `qwen/qwen3-coder-30b-a3b:instruct` | OpenRouter | `OPENROUTER_API_KEY` |

Create a `.env` file in the same directory as this notebook:
```
OPENAI_API_KEY=sk-...
OPENROUTER_API_KEY=sk-or-...
GROQ_API_KEY=gsk_...
GROK_API_KEY=xai-...
```

```bash
# Ollama quick-start if  not installed already
brew install ollama          # or https://ollama.com
ollama pull qwen2.5-coder
ollama serve
```